# Ⅰ第2回 演習3 用途に合わせてモデルを選ぶ

演習2 で測った 3 モデル（MLP・CNN・Transformer）の数値を見て，**用途に合わせてどのモデルを選ぶか** をグループで決める．
点数の計算はしない．数値を見て話し合い，選んだ理由を言葉にする．

演習2 と同じく，全員が自分のノートパソコンで「✋ 止まる」でそろいながら進む．

## (0) グループと役割の設定

In [1]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 12                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "verifier"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

作業フォルダ: /Users/hinata/AI_TD


## (1) ライブラリのインポート

In [2]:
import pandas as pd
from common import RESULTS
from common.logger import ResultLogger, read_results
from common.device import get_device
from common.todo import todo_check, checkpoint
device = get_device()

## (2) 演習2 の結果を見る

自分が演習2 で測った値を表にする（速さとメモリは，マシンによって違う）．

In [3]:
df = read_results(course="c1", day="d2", exercise="ex2")
df = df[(df["group_id"] == f"{GROUP_ID:02d}") & df["condition"].isin(["MLP", "CNN", "Transformer"])]
if not (len(df) > 0):
    raise RuntimeError("演習2 の結果がない．先に ex2.ipynb を最後まで実行すること")
df = df.sort_values("timestamp").groupby(["condition", "metric_name"]).tail(1)   # 何回も実行していれば最新の値
table = df.pivot(index="condition", columns="metric_name", values="metric_value")
JA = {"accuracy": "正解率", "loss": "損失", "infer_ms_per_1000": "1000文の時間(ms)",
      "latency_ms_b1": "1文の時間(ms)", "peak_mem_mb": "推論中のピーク(MB)"}
table = table[list(JA)].rename(columns=JA).rename_axis("モデル")
table.round(4)

metric_name,正解率,損失,1000文の時間(ms),1文の時間(ms),推論中のピーク(MB)
モデル,,,,,
CNN,0.7897,0.6941,5.1051,0.3351,3.00
MLP,0.8056,0.4664,3.1736,0.2020,0.32
Transformer,0.8094,0.4348,72.3154,1.1028,11.33


## ✋ 止まる 1：ここから先へは，グループ全員がそろってから進む

**問い**：3 つのモデルは，それぞれどの指標で一番よく，どの指標で一番悪いか

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(2) の表の各列で，一番よい値と悪い値．正解率は高いほど，それ以外は低いほどよい．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP1 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [4]:
# 問い：3 つのモデルは，それぞれどの指標で一番よく，どの指標で一番悪いか
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP1 = "〇〇だから，△△になると考えた．"
#   注意：CP1 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP1 = "CNNは1文の時間が一番よく,正解率が一番悪いと考えた。MLPは1,000文の時間が一番よく,推論中のピークが一番悪いと考えた。Transformerは正解率が一番よく,1000文の時間が一番悪いと考えた。"
checkpoint(CP1, "止まる 1")

止まる 1 OK：CNNは1文の時間が一番よく,正解率が一番悪いと考えた。MLPは1,000文の時間が一番よく,推論中のピークが一番悪いと考えた。Transformerは正解率が一番よく,1000文の時間が一番悪いと考えた。


## (3) 討議（10 分）：用途を決めて，モデルを選ぶ

1. 各自 1 分：「このモデルを **誰が，どこで** 使うか」を 1 つ挙げる
2. グループで用途を 1 つに絞る（割れたら「その用途で失敗するとどうなるか」で決める）
3. その用途で **一番大事な指標** と，**我慢できる指標** を決める
4. (2) の表を見て，どのモデルを選ぶか決める．理由を，表の数値を使って言えるようにする

例：「学生のスマホで動く映画レビュー判定器」なら，1 文の時間とメモリが大事．正解率は 1 ポイント程度なら我慢できる．

In [ ]:
# ▼ TODO：グループで決めたものを " " の中に書く（全員が同じものを書く）
USE = ""            # 用途（誰が，どこで使うか）
IMPORTANT = ""      # 一番大事な指標と，我慢できる指標
CHOICE = ""         # 選んだモデル："MLP" / "CNN" / "Transformer" のどれか
REASON = ""         # 選んだ理由（表の数値を使って 1〜2 文）

if CHOICE not in ("MLP", "CNN", "Transformer"):
    raise ValueError('CHOICE は "MLP" / "CNN" / "Transformer" のどれかにする')
for name, text in [("USE", USE), ("IMPORTANT", IMPORTANT), ("REASON", REASON)]:
    checkpoint(text, name, min_chars=5)

## (4) 用途が変わると，選ぶモデルは変わるか

下の 3 つの用途なら，どのモデルを選ぶか．グループで話して書く（1 つ 1 分）．

In [ ]:
# ▼ TODO：それぞれの用途で選ぶモデルを書く（"MLP" / "CNN" / "Transformer"）
OTHER_USES = {
    "研究で，少しでも正解率の高いモデルを報告したい": "",
    "1 日 1 億件のレビューをサーバでまとめて判定したい": "",
    "メモリの少ない小型の機械に組み込んで動かしたい": "",
}
for use, model in OTHER_USES.items():
    if model not in ("MLP", "CNN", "Transformer"):
        raise ValueError(f"「{use}」で選ぶモデルを書く")
summary = pd.DataFrame({"用途": [USE] + list(OTHER_USES), "選んだモデル": [CHOICE] + list(OTHER_USES.values())})
summary

## ✋ 止まる 2：ここから先へは，グループ全員がそろってから進む

**問い**：用途によって選ぶモデルは変わったか．どの数値の違いが，選ぶモデルを変えたか

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(2) の表で，用途ごとに注目した列と，その列でのモデルの差．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP2 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [ ]:
# 問い：用途によって選ぶモデルは変わったか．どの数値の違いが，選ぶモデルを変えたか
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP2 = "〇〇だから，△△になると考えた．"
#   注意：CP2 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP2 = ""
checkpoint(CP2, "止まる 2")

## (5) 記録

選んだモデルと理由を残す．

In [ ]:
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d2", exercise="ex3", device=device)
logger.log("chosen", 1.0, condition=CHOICE)
for use, model in OTHER_USES.items():
    logger.log("chosen_other", 1.0, condition=model)
(RESULTS / f"c1_d2_ex3_choice_{logger.group_id}.txt").write_text(
    f"用途: {USE}\n大事な指標: {IMPORTANT}\n選んだモデル: {CHOICE}\n理由: {REASON}\n", encoding="utf-8")
print("書き込み先:", logger.path)

## (6) グループ内プレゼン用まとめ

スライドは作らない．この出力を画面に出して話す（話す内容は授業ページ演習3）．

In [ ]:
print("グループ", logger.group_id)
print("用途　　　:", USE)
print("大事な指標:", IMPORTANT)
print("選んだモデル:", CHOICE)
print("理由　　　:", REASON)
display(table.round(4))

## (7) 全体共有

教員が各グループの用途と選んだモデルを並べて投影する．**同じ 3 モデルの数値から，用途によって違うモデルが選ばれる** ことを確かめる．

**第4回に持ち越す問い**：正解率の差 0.4 ポイント（MLP と Transformer）は，本当に「差がある」と言えるか．ばらつきの範囲内ではないか．